# Custom Tools & Advanced Schemas

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) LangChain roadmap.

You will learn how to define complex tool schemas, handle errors, and build state-updating tools.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain "langchain[openai]" langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Pydantic args_schema

Define a Pydantic model as `args_schema` for detailed parameter descriptions.

In [ ]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from typing import Optional

class SearchInput(BaseModel):
    query: str = Field(description="The search query string")
    max_results: int = Field(default=5, description="Maximum results to return")
    category: Optional[str] = Field(default=None, description="Filter by category")

@tool(args_schema=SearchInput)
def advanced_search(query: str, max_results: int = 5, category: Optional[str] = None) -> str:
    """Search for items with advanced filtering."""
    return f"Found {max_results} results for '{query}' in {category or 'all categories'}"

print("Name:", advanced_search.name)
print("Schema:", advanced_search.args_schema.model_json_schema())

## 4. Complex Input Types

Use enums, lists, and nested objects for rich schemas.

In [ ]:
from typing import List
from enum import Enum

class Priority(str, Enum):
    low = "low"
    medium = "medium"
    high = "high"

class TaskInput(BaseModel):
    title: str = Field(description="Task title")
    priority: Priority = Field(description="Task priority level")
    tags: List[str] = Field(default_factory=list, description="List of tags")

@tool(args_schema=TaskInput)
def create_task(title: str, priority: Priority, tags: List[str] = []) -> str:
    """Create a new task."""
    return f"Created '{title}' (priority: {priority.value}, tags: {tags})"

print(create_task.invoke({"title": "Fix bug", "priority": "high", "tags": ["backend", "urgent"]}))

## 5. ToolNode for LangGraph

`ToolNode` is a prebuilt node that handles tool execution in LangGraph.

In [ ]:
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessage, ToolCall

tool_node = ToolNode([advanced_search, create_task])

message = AIMessage(
    content="",
    tool_calls=[
        ToolCall(
            id="call_1",
            name="advanced_search",
            args={"query": "python tutorials", "max_results": 3},
        )
    ],
)

result = tool_node.invoke({"messages": [message]})
print(result["messages"][-1].content)

## 6. handle_tool_errors

Enable error handling so the agent can recover from tool failures.

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

model = init_chat_model("gpt-4o-mini", model_provider="openai")

@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b

agent = create_react_agent(
    model=model,
    tools=[divide],
    tool_node_kwargs={"handle_tool_errors": True},
)

result = agent.invoke({
    "messages": [HumanMessage(content="What is 10 divided by 0?")]
})
print(result["messages"][-1].content)

## 7. State-Updating Tools with Command

Tools can update the agent's state directly using `Command`.

In [ ]:
from langgraph.types import Command
from langchain_core.messages import SystemMessage

@tool
def set_language(language: str) -> Command:
    """Set the preferred response language."""
    return Command(
        update={
            "messages": [SystemMessage(content=f"Respond in {language} from now on.")]
        }
    )

agent = create_react_agent(
    model=model,
    tools=[set_language],
)

result = agent.invoke({
    "messages": [HumanMessage(content="Set language to Spanish, then greet me.")]
})
print(result["messages"][-1].content)

## 8. Using Tools with the Agent

Combine custom tools with an agent for a complete workflow.

In [ ]:
agent = create_react_agent(
    model=model,
    tools=[advanced_search, create_task, divide],
    prompt="You are a project assistant. Use tools to help manage tasks and answer questions.",
    tool_node_kwargs={"handle_tool_errors": True},
)

result = agent.invoke({
    "messages": [HumanMessage(content="Search for 'API design patterns' and create a high priority task to review the results")]
})

for msg in result["messages"]:
    if msg.type == "tool":
        print(f"Tool [{msg.name}]: {msg.content}")
print(f"\nAgent: {result['messages'][-1].content}")

## Summary

- Use `args_schema` with Pydantic for complex tool inputs
- Pydantic supports enums, lists, nested objects, and optional fields
- `ToolNode` executes tools in LangGraph workflows
- `handle_tool_errors` lets agents recover from tool failures
- `Command` allows tools to update agent state directly